In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
p = pathlib.Path.cwd()
for q in (p, *p.parents):
    s = q / "src" / "ftbp"   # <- change "ftbp" if you rename the package
    if s.exists():
        sys.path.insert(0, str(s.parent))  # add .../src
        break
else:
    raise RuntimeError("src/ftbp not found")

In [ ]:
import numpy as np
import pandas as pd
import itertools
from math import comb
from scipy.optimize import brentq
from scipy.stats import norm, cauchy, uniform
from scipy.stats import median_abs_deviation
from ftbp.bootstrap import *
from ftbp.bootstrap import _compute_basics_bootstrap
from ftbp.io import *
import seaborn as sns
import matplotlib.pyplot as plt

## Coverage Tests

In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.integrate import quad
from scipy.optimize import brentq
# ----------------------------
# Conditional E[psi(X - theta) | X >= q_alpha] via quad
# ----------------------------
def E_cond_upper_numeric(theta, alpha, psi, psi_kwargs=None, epsabs=1e-10, epsrel=1e-8):
    if psi_kwargs is None:
        psi_kwargs = {}
    a = norm.ppf(alpha)            # q_alpha
    D = 1.0 - norm.cdf(a)          # tail mass
    if D <= 0: 
        return 0.0
    integrand = lambda x: psi(x - theta, **psi_kwargs) * norm.pdf(x)
    num, _ = quad(integrand, a, np.inf, epsabs=epsabs, epsrel=epsrel, limit=200)
    return num / D

# ----------------------------
# Closed-form for Huber (fast, no quad)
# ----------------------------
def E_cond_upper_huber_cf(theta, alpha, delta):
    a = norm.ppf(alpha)      # truncation point
    D = 1.0 - norm.cdf(a)    # tail mass
    t1, t2 = theta - delta, theta + delta

    # region 1: x in [a, t1] => psi = -delta
    if t1 > a:
        I_low = -delta * (norm.cdf(t1) - norm.cdf(a))
    else:
        I_low = 0.0

    # region 2: x in [max(a, t1), t2] => psi = x - theta, provided t2 > max(a,t1)
    L = max(a, t1)
    U = t2
    if U > L:
        # ∫(x-θ) φ(x) dx = [-φ(x) - θ Φ(x)]_L^U
        I_mid = (-norm.pdf(U) - theta * norm.cdf(U)) - (-norm.pdf(L) - theta * norm.cdf(L))
    else:
        I_mid = 0.0

    # region 3: x in [max(a, t2), ∞) => psi = +delta
    U2 = max(a, t2)
    I_up = delta * (1.0 - norm.cdf(U2))

    return (I_low + I_mid + I_up) / D

# ----------------------------
# Root solver for upward push
# ----------------------------
def solve_theta_alpha_plus(alpha, psi, delta, psi_kwargs=None,
                           use_huber_closed_form=False, 
                           bracket=(0.0, 1.0), tol=1e-10, max_expand=60):
    if psi_kwargs is None:
        psi_kwargs = {}

    # choose expectation engine
    if use_huber_closed_form:
        Econd = lambda th: E_cond_upper_huber_cf(th, alpha, psi_kwargs.get("delta", delta))
    else:
        Econd = lambda th: E_cond_upper_numeric(th, alpha, psi, psi_kwargs)

    # g(theta) = (1-alpha) E[psi(X-theta) | X>=q_alpha] + alpha*delta
    g = lambda th: (1.0 - alpha) * Econd(th) + alpha * delta

    lo, hi = bracket
    g_lo, g_hi = g(lo), g(hi)
    # Expand upper bracket until sign change (for alpha<0.5 this will happen)
    k = 0
    while g_hi > 0 and k < max_expand:
        hi = hi * 2.0 if hi > 0 else 1.0
        g_hi = g(hi); k += 1
    if g_lo * g_hi > 0:
        raise RuntimeError(f"Failed to bracket root: g({lo})={g_lo}, g({hi})={g_hi}")

    return brentq(g, lo, hi, xtol=tol, rtol=tol, maxiter=2000)

# ----------------------------
# Convenience wrapper to get eta_alpha
# (for symmetric F and odd psi: eta_alpha = theta_alpha_plus)
# ----------------------------
def eta_alpha(alpha, psi='huber', delta=1.345, **kwargs):
    if psi == 'huber':
        theta = solve_theta_alpha_plus(alpha, psi_prime, delta, psi_kwargs={'delta': delta},
                                       use_huber_closed_form=True, **kwargs)
    else:
        raise NotImplementedError("Add your psi(u) and pass psi callable + psi_kwargs.")
    return theta

# ---- Example ----
for a in [0.03, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
    th = eta_alpha(a, psi='huber', delta=1.345)
    print('a:', a, 'eta:', th)


In [ ]:
# Coverage test
n = 100
B = 1000
M = 1000
m = 3

coverage_80 = []
coverage_95 = []
np.random.seed(53)

eta_true = 0.09833365678787231

for _ in range(M):
    # print(eta_true)
    # true data
    x = np.random.normal(size=n)
    bootstrap_lst = []
    for _ in range(B):
        wx = np.random.exponential(scale=1.0, size=len(x))
        theta_hat = estimate_theta_bootstrap(x, wx, delta=1.345, loss_type='huber')
        eta_hat = eta_theta_plus(x, wx, theta_hat, m=m, delta=1.345, loss_type='huber')
        bootstrap_lst.append(eta_hat)
    bootstrap_lst = np.array(bootstrap_lst)
    # print(bootstrap_lst)
    quantile_025 = np.percentile(bootstrap_lst, 2.5)
    quantile_975 = np.percentile(bootstrap_lst, 97.5)
    quantile_10 = np.percentile(bootstrap_lst, 10)
    quantile_90 = np.percentile(bootstrap_lst, 90)
    coverage_95.append(int((quantile_025 <= eta_true) and (eta_true <= quantile_975)))
    coverage_80.append(int((quantile_10 <= eta_true) and (eta_true <= quantile_90)))

print(f"Coverage 95%: {np.mean(np.array(coverage_95)):.3f}")
print(f"Coverage 80%: {np.mean(np.array(coverage_80)):.3f}")


In [ ]:
# Coverage test
n = 100
B = 1000
M = 1000
m = 10

coverage_80 = []
coverage_95 = []
np.random.seed(53)

eta_true = 0.3306654573280327

for _ in range(M):
    # print(eta_true)
    # true data
    x = np.random.normal(size=n)
    bootstrap_lst = []
    for _ in range(B):
        wx = np.random.exponential(scale=1.0, size=len(x))
        theta_hat = estimate_theta_bootstrap(x, wx, delta=1.345, loss_type='huber')
        eta_hat = eta_theta_plus(x, wx, theta_hat, m=m, delta=1.345, loss_type='huber')
        bootstrap_lst.append(eta_hat)
    bootstrap_lst = np.array(bootstrap_lst)
    # print(bootstrap_lst)
    quantile_025 = np.percentile(bootstrap_lst, 2.5)
    quantile_975 = np.percentile(bootstrap_lst, 97.5)
    quantile_10 = np.percentile(bootstrap_lst, 10)
    quantile_90 = np.percentile(bootstrap_lst, 90)
    # print(quantile_025, quantile_975, quantile_10, quantile_90, eta_true)
    coverage_95.append(int((quantile_025 <= eta_true) and (eta_true <= quantile_975)))
    coverage_80.append(int((quantile_10 <= eta_true) and (eta_true <= quantile_90)))

print(f"Coverage 95%: {np.mean(np.array(coverage_95)):.3f}")
print(f"Coverage 80%: {np.mean(np.array(coverage_80)):.3f}")


In [ ]:
# Coverage test
n = 100
B = 1000
M = 1000
m = 15

coverage_80 = []
coverage_95 = []
np.random.seed(53)

eta_true = 0.5022481166353134

for _ in range(M):
    # print(eta_true)
    # true data
    x = np.random.normal(size=n)
    bootstrap_lst = []
    for _ in range(B):
        wx = np.random.exponential(scale=1.0, size=len(x))
        theta_hat = estimate_theta_bootstrap(x, wx, delta=1.345, loss_type='huber')
        eta_hat = eta_theta_plus(x, wx, theta_hat, m=m, delta=1.345, loss_type='huber')
        bootstrap_lst.append(eta_hat)
    bootstrap_lst = np.array(bootstrap_lst)
    # print(bootstrap_lst)
    quantile_025 = np.percentile(bootstrap_lst, 2.5)
    quantile_975 = np.percentile(bootstrap_lst, 97.5)
    quantile_10 = np.percentile(bootstrap_lst, 10)
    quantile_90 = np.percentile(bootstrap_lst, 90)
    # print(quantile_025, quantile_975, quantile_10, quantile_90, eta_true)
    coverage_95.append(int((quantile_025 <= eta_true) and (eta_true <= quantile_975)))
    coverage_80.append(int((quantile_10 <= eta_true) and (eta_true <= quantile_90)))

print(f"Coverage 95%: {np.mean(np.array(coverage_95)):.3f}")
print(f"Coverage 80%: {np.mean(np.array(coverage_80)):.3f}")

In [ ]:
from scipy.stats import kstest
sns.set_style("whitegrid")    # e.g. “darkgrid”, “ticks”, “white”
sns.set_context("talk")       # options: “paper”, “notebook”, “talk”, “poster”
palette = sns.color_palette("husl", 3)
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})

In [ ]:
# comparison with pure uniform data
np.random.seed(53)
x = np.random.uniform(size=3000)
plt.hist(x, bins=20, alpha=0.5, density=True, label="uniform data")
plt.plot([0, 1], [1, 1], color='black', linestyle='--', label='uniform')

In [ ]:
# a bootstrap plot showing uniform
n = 100
B = 1000
M = 1000
m = 3

sampling_law = []
bootstrap_law = []
np.random.seed(53)

eta_true_lst = []
eta_true = 0.09833366714275775

for _ in range(M):
    x = np.random.normal(size=n)
    eta_x = eta_theta_plus(x, np.ones(len(x)), estimate_theta(x, delta=1.345, loss_type='huber'), m=m, delta=1.345, loss_type='huber')
    sampling_law.append(np.sqrt(n) * (eta_x - eta_true))
    bootstrap_lst = []
    for _ in range(B):
        wx = np.random.exponential(scale=1.0, size=len(x))
        # wx /= np.mean(wx)
        # wx = np.random.dirichlet(np.ones(len(x)), size=1).flatten() * len(x)
        theta_hat = estimate_theta_bootstrap(x, wx, delta=1.345, loss_type='huber')
        eta_hat = eta_theta_plus(x, wx, theta_hat, m=m, delta=1.345, loss_type='huber')
        bootstrap_lst.append(eta_hat)
    bootstrap_lst = np.array(bootstrap_lst) - eta_x 
    bootstrap_law.append(np.sort(bootstrap_lst) * np.sqrt(n))
    
sampling_law = np.array(sampling_law)
sampling_law = np.sort(sampling_law)

# B: PIT (randomized if discrete)
U = []
for j, T_j in enumerate(sampling_law):
    fs = bootstrap_law[j]
    # randomized PIT for discrete: compute left-limit if needed; else plain rank/B
    U.append((np.searchsorted(fs, T_j, side='right') + np.random.uniform())/(B+1))

D, p_value = kstest(U, 'uniform')
print(f"KS statistic: {D:.3f}, p-value: {p_value:.3f}")
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.plot(np.linspace(0, 1, 1000), np.linspace(0, 1, 1000), color='blue', label='y=x')
plt.plot(np.linspace(0, 1, 1000), np.quantile(U, np.linspace(0, 1, 1000)), color='red', label='PP plot')
plt.xlabel("Theoretical quantiles")
plt.ylabel("Empirical quantiles")
plt.title(f"PP plot over {M} trials")
# annotate with ks stat and p-value
plt.text(0.55, 0, f"KS stat: {D:.3f}\np-value: {p_value:.3f}", bbox=dict(facecolor='white', alpha=0.5))
plt.legend()
plt.subplot(1, 2, 2)
plt.hist(U, bins=20, alpha=0.5, density=True, label=f"Ratio={m/n:.2f}")
plt.plot([0, 1], [1, 1], color='black', linestyle='--', label='Uniform')
plt.xlabel("Quantile")
plt.ylabel("Density")
# legend to lower left
plt.legend(loc='lower left')
plt.title(f"Histogram of quantiles over {M} trials")
plt.tight_layout()
plt.savefig(f"bootstrap_uniformity_ppplot_m_{m}_n_{n}.pdf")

In [ ]:
# a bootstrap plot showing uniform
n = 1000
B = 3000
M = 3000
m = 30

sampling_law = []
bootstrap_law = []
np.random.seed(53)

eta_true_lst = []
eta_true = 0.09833366714275775

for _ in range(M):
    x = np.random.normal(size=n)
    eta_x = eta_theta_plus(x, np.ones(len(x)), estimate_theta(x, delta=1.345, loss_type='huber'), m=m, delta=1.345, loss_type='huber')
    sampling_law.append(np.sqrt(n) * (eta_x - eta_true))
    bootstrap_lst = []
    for _ in range(B):
        wx = np.random.exponential(scale=1.0, size=len(x))
        # wx /= np.mean(wx)
        # wx = np.random.dirichlet(np.ones(len(x)), size=1).flatten() * len(x)
        theta_hat = estimate_theta_bootstrap(x, wx, delta=1.345, loss_type='huber')
        eta_hat = eta_theta_plus(x, wx, theta_hat, m=m, delta=1.345, loss_type='huber')
        bootstrap_lst.append(eta_hat)
    bootstrap_lst = np.array(bootstrap_lst) - eta_x 
    bootstrap_law.append(np.sort(bootstrap_lst) * np.sqrt(n))
    
sampling_law = np.array(sampling_law)
sampling_law = np.sort(sampling_law)

# B: PIT (randomized if discrete)
U = []
for j, T_j in enumerate(sampling_law):
    fs = bootstrap_law[j]
    # randomized PIT for discrete: compute left-limit if needed; else plain rank/B
    U.append((np.searchsorted(fs, T_j, side='right') + np.random.uniform())/(B+1))

D, p_value = kstest(U, 'uniform')
print(f"KS statistic: {D:.3f}, p-value: {p_value:.3f}")
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.plot(np.linspace(0, 1, 1000), np.linspace(0, 1, 1000), color='blue', label='y=x')
plt.plot(np.linspace(0, 1, 1000), np.quantile(U, np.linspace(0, 1, 1000)), color='red', label='PP plot')
plt.xlabel("Theoretical quantiles")
plt.ylabel("Empirical quantiles")
plt.title(f"PP plot over {M} trials")
# annotate with ks stat and p-value
plt.text(0.55, 0, f"KS stat: {D:.3f}\np-value: {p_value:.3f}", bbox=dict(facecolor='white', alpha=0.5))
plt.legend()
plt.subplot(1, 2, 2)
plt.hist(U, bins=20, alpha=0.5, density=True, label=f"Ratio={m/n:.2f}")
plt.plot([0, 1], [1, 1], color='black', linestyle='--', label='Uniform')
plt.xlabel("Quantile")
plt.ylabel("Density")
# legend to lower left
plt.legend(loc='lower left')
plt.title(f"Histogram of quantiles over {M} trials")
plt.tight_layout()
plt.savefig(f"bootstrap_uniformity_ppplot_m_{m}_n_{n}.pdf")

In [ ]:
# a bootstrap plot showing uniform
n = 100
B = 1000
M = 1000
m = 10

sampling_law = []
bootstrap_law = []
np.random.seed(53)

eta_true_lst = []
eta_true = 0.3306654573280327

for _ in range(M):
    x = np.random.normal(size=n)
    eta_x = eta_theta_plus(x, np.ones(len(x)), estimate_theta(x, delta=1.345, loss_type='huber'), m=m, delta=1.345, loss_type='huber')
    sampling_law.append(np.sqrt(n) * (eta_x - eta_true))
    bootstrap_lst = []
    for _ in range(B):
        wx = np.random.exponential(scale=1.0, size=len(x))
        theta_hat = estimate_theta_bootstrap(x, wx, delta=1.345, loss_type='huber')
        eta_hat = eta_theta_plus(x, wx, theta_hat, m=m, delta=1.345, loss_type='huber')
        bootstrap_lst.append(eta_hat)
    bootstrap_lst = np.array(bootstrap_lst) - eta_x 
    bootstrap_law.append(np.sort(bootstrap_lst) * np.sqrt(n))
    
sampling_law = np.array(sampling_law)
sampling_law = np.sort(sampling_law)

# B: PIT (randomized if discrete)
U = []
for j, T_j in enumerate(sampling_law):
    fs = bootstrap_law[j]
    # randomized PIT for discrete: compute left-limit if needed; else plain rank/B
    U.append((np.searchsorted(fs, T_j, side='right') + np.random.uniform())/(B+1))

D, p_value = kstest(U, 'uniform')
print(f"KS statistic: {D:.3f}, p-value: {p_value:.3f}")
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.plot(np.linspace(0, 1, 1000), np.linspace(0, 1, 1000), color='blue', label='y=x')
plt.plot(np.linspace(0, 1, 1000), np.quantile(U, np.linspace(0, 1, 1000)), color='red', label='PP plot')
plt.xlabel("Theoretical quantiles")
plt.ylabel("Empirical quantiles")
plt.title(f"PP plot over {M} trials")
# annotate with ks stat and p-value
plt.text(0.55, 0, f"KS stat: {D:.3f}\np-value: {p_value:.3f}", bbox=dict(facecolor='white', alpha=0.5))
plt.legend()
plt.subplot(1, 2, 2)
plt.hist(U, bins=20, alpha=0.5, density=True, label=f"Ratio={m/n:.2f}")
plt.plot([0, 1], [1, 1], color='black', linestyle='--', label='Uniform')
plt.xlabel("Quantile")
plt.ylabel("Density")
# legend to lower left
plt.legend(loc='lower left')
plt.title(f"Histogram of quantiles over {M} trials")
plt.tight_layout()
plt.savefig(f"bootstrap_uniformity_ppplot_m_{m}_n_{n}.pdf")

In [ ]:
# a bootstrap plot showing uniform
n = 1000
B = 3000
M = 3000
m = 100

sampling_law = []
bootstrap_law = []
np.random.seed(53)

eta_true_lst = []
eta_true = 0.3306654573280327

for _ in range(M):
    x = np.random.normal(size=n)
    eta_x = eta_theta_plus(x, np.ones(len(x)), estimate_theta(x, delta=1.345, loss_type='huber'), m=m, delta=1.345, loss_type='huber')
    sampling_law.append(np.sqrt(n) * (eta_x - eta_true))
    bootstrap_lst = []
    for _ in range(B):
        wx = np.random.exponential(scale=1.0, size=len(x))
        theta_hat = estimate_theta_bootstrap(x, wx, delta=1.345, loss_type='huber')
        eta_hat = eta_theta_plus(x, wx, theta_hat, m=m, delta=1.345, loss_type='huber')
        bootstrap_lst.append(eta_hat)
    bootstrap_lst = np.array(bootstrap_lst) - eta_x 
    bootstrap_law.append(np.sort(bootstrap_lst) * np.sqrt(n))
    
sampling_law = np.array(sampling_law)
sampling_law = np.sort(sampling_law)

# B: PIT (randomized if discrete)
U = []
for j, T_j in enumerate(sampling_law):
    fs = bootstrap_law[j]
    # randomized PIT for discrete: compute left-limit if needed; else plain rank/B
    U.append((np.searchsorted(fs, T_j, side='right') + np.random.uniform())/(B+1))

D, p_value = kstest(U, 'uniform')
print(f"KS statistic: {D:.3f}, p-value: {p_value:.3f}")
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.plot(np.linspace(0, 1, 1000), np.linspace(0, 1, 1000), color='blue', label='y=x')
plt.plot(np.linspace(0, 1, 1000), np.quantile(U, np.linspace(0, 1, 1000)), color='red', label='PP plot')
plt.xlabel("Theoretical quantiles")
plt.ylabel("Empirical quantiles")
plt.title(f"PP plot over {M} trials")
# annotate with ks stat and p-value
plt.text(0.55, 0, f"KS stat: {D:.3f}\np-value: {p_value:.3f}", bbox=dict(facecolor='white', alpha=0.5))
plt.legend()
plt.subplot(1, 2, 2)
plt.hist(U, bins=20, alpha=0.5, density=True, label=f"Ratio={m/n:.2f}")
plt.plot([0, 1], [1, 1], color='black', linestyle='--', label='Uniform')
plt.xlabel("Quantile")
plt.ylabel("Density")
# legend to lower left
plt.legend(loc='lower left')
plt.title(f"Histogram of quantiles over {M} trials")
plt.tight_layout()
plt.savefig(f"bootstrap_uniformity_ppplot_m_{m}_n_{n}.pdf")

In [ ]:
# a bootstrap plot showing uniform
n = 100
B = 1000
M = 1000
m = 15

sampling_law = []
bootstrap_law = []
np.random.seed(53)

eta_true_lst = []
eta_true = 0.5022481166353134

for _ in range(M):
    x = np.random.normal(size=n)
    eta_x = eta_theta_plus(x, np.ones(len(x)), estimate_theta(x, delta=1.345, loss_type='huber'), m=m, delta=1.345, loss_type='huber')
    sampling_law.append(np.sqrt(n) * (eta_x - eta_true))
    bootstrap_lst = []
    for _ in range(B):
        wx = np.random.exponential(scale=1.0, size=len(x))
        theta_hat = estimate_theta_bootstrap(x, wx, delta=1.345, loss_type='huber')
        eta_hat = eta_theta_plus(x, wx, theta_hat, m=m, delta=1.345, loss_type='huber')
        bootstrap_lst.append(eta_hat)
    bootstrap_lst = np.array(bootstrap_lst) - eta_x 
    bootstrap_law.append(np.sort(bootstrap_lst) * np.sqrt(n))
    
sampling_law = np.array(sampling_law)
sampling_law = np.sort(sampling_law)

# B: PIT (randomized if discrete)
U = []
for j, T_j in enumerate(sampling_law):
    fs = bootstrap_law[j]
    # randomized PIT for discrete: compute left-limit if needed; else plain rank/B
    U.append((np.searchsorted(fs, T_j, side='right') + np.random.uniform())/(B+1))

D, p_value = kstest(U, 'uniform')
print(f"KS statistic: {D:.3f}, p-value: {p_value:.3f}")
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.plot(np.linspace(0, 1, 1000), np.linspace(0, 1, 1000), color='blue', label='y=x')
plt.plot(np.linspace(0, 1, 1000), np.quantile(U, np.linspace(0, 1, 1000)), color='red', label='PP plot')
plt.xlabel("Theoretical quantiles")
plt.ylabel("Empirical quantiles")
plt.title(f"PP plot over {M} trials")
# annotate with ks stat and p-value
plt.text(0.55, 0, f"KS stat: {D:.3f}\np-value: {p_value:.3f}", bbox=dict(facecolor='white', alpha=0.5))
plt.legend()
plt.subplot(1, 2, 2)
plt.hist(U, bins=20, alpha=0.5, density=True, label=f"Ratio={m/n:.2f}")
plt.plot([0, 1], [1, 1], color='black', linestyle='--', label='Uniform')
plt.xlabel("Quantile")
plt.ylabel("Density")
# legend to lower left
plt.legend(loc='lower left')
plt.title(f"Histogram of quantiles over {M} trials")
plt.tight_layout()
plt.savefig(f"bootstrap_uniformity_ppplot_m_{m}_n_{n}.pdf")

In [ ]:
# a bootstrap plot showing uniform
n = 1000
B = 3000
M = 3000
m = 150

sampling_law = []
bootstrap_law = []
np.random.seed(53)

eta_true_lst = []
eta_true = 0.5022481166353134

for _ in range(M):
    x = np.random.normal(size=n)
    eta_x = eta_theta_plus(x, np.ones(len(x)), estimate_theta(x, delta=1.345, loss_type='huber'), m=m, delta=1.345, loss_type='huber')
    sampling_law.append(np.sqrt(n) * (eta_x - eta_true))
    bootstrap_lst = []
    for _ in range(B):
        wx = np.random.exponential(scale=1.0, size=len(x))
        theta_hat = estimate_theta_bootstrap(x, wx, delta=1.345, loss_type='huber')
        eta_hat = eta_theta_plus(x, wx, theta_hat, m=m, delta=1.345, loss_type='huber')
        bootstrap_lst.append(eta_hat)
    bootstrap_lst = np.array(bootstrap_lst) - eta_x 
    bootstrap_law.append(np.sort(bootstrap_lst) * np.sqrt(n))
    
sampling_law = np.array(sampling_law)
sampling_law = np.sort(sampling_law)

# B: PIT (randomized if discrete)
U = []
for j, T_j in enumerate(sampling_law):
    fs = bootstrap_law[j]
    # randomized PIT for discrete: compute left-limit if needed; else plain rank/B
    U.append((np.searchsorted(fs, T_j, side='right') + np.random.uniform())/(B+1))
    
D, p_value = kstest(U, 'uniform')
print(f"KS statistic: {D:.3f}, p-value: {p_value:.3f}")
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.plot(np.linspace(0, 1, 1000), np.linspace(0, 1, 1000), color='blue', label='y=x')
plt.plot(np.linspace(0, 1, 1000), np.quantile(U, np.linspace(0, 1, 1000)), color='red', label='PP plot')
plt.xlabel("Theoretical quantiles")
plt.ylabel("Empirical quantiles")
plt.title(f"PP plot over {M} trials")
# annotate with ks stat and p-value
plt.text(0.55, 0, f"KS stat: {D:.3f}\np-value: {p_value:.3f}", bbox=dict(facecolor='white', alpha=0.5))
plt.legend()
plt.subplot(1, 2, 2)
plt.hist(U, bins=20, alpha=0.5, density=True, label=f"Ratio={m/n:.2f}")
plt.plot([0, 1], [1, 1], color='black', linestyle='--', label='Uniform')
plt.xlabel("Quantile")
plt.ylabel("Density")
# legend to lower left
plt.legend(loc='lower left')
plt.title(f"Histogram of quantiles over {M} trials")
plt.tight_layout()
plt.savefig(f"bootstrap_uniformity_ppplot_m_{m}_n_{n}.pdf")

## Bootstrap Bands

In [ ]:
def bootstrap_band(eta_grid, n1=100, n2=100, B=100,
                   delta=1.345, alpha=0.05, loss_type='huber', seed=53):
    rows = []
    for eta in eta_grid:
        np.random.seed(seed)
        x_s = np.random.normal(loc=0.0, scale=1.0, size=n1)
        y_s = np.random.normal(loc=1, scale=1.0, size=n2)
        wx = np.ones(len(x_s), dtype=float); wy = np.ones(len(y_s), dtype=float)
        bp = bp_at_eta_bootstrap(x_s, y_s, wx, wy, eta=eta, delta=delta, alpha=alpha, loss_type=loss_type)['bp']
        boot_bp = []
        boot_wbp = []
        for b in range(B):
            # x_b = np.random.choice(x_s, size=n1, replace=True)
            # y_b = np.random.choice(y_s, size=n2, replace=True)
            # bps = bp_at_eta_bootstrap(x_b, y_b, np.ones(len(x_b), dtype=float), np.ones(len(y_b), dtype=float), eta=eta, delta=delta, alpha=alpha, 
            # loss_type=loss_type)
            wxb = np.random.exponential(scale=1.0, size=len(x_s))
            wyb = np.random.exponential(scale=1.0, size=len(y_s))
            bps = bp_at_eta_bootstrap(x_s, y_s, wxb, wyb, eta=eta, delta=delta, alpha=alpha, loss_type=loss_type)
            bpb = bps['bp']
            boot_bp.append(bpb)
        boot_bp = np.array(boot_bp)
        rows.append({"eta": float(eta), "bp": float(bp),
                     "bp_lower": float(np.percentile(boot_bp, 2.5)),
                     "bp_upper": float(np.percentile(boot_bp, 97.5)),
                     "bpb": boot_bp,
                     "n1": n1, "n2": n2, "delta": delta, "alpha": alpha, 
                     "loss": loss_type, "B": B})
    df = pd.DataFrame(rows)
    return df

In [ ]:
# Do some simulation
n1 = 100
n2 = 100
B = 1000
etas = [.1, .25, .5, .75, 1, 1.25, 1.5, 1.75, 2]
# loss_types = ['huber', 'logcosh', 'concordant']
loss_types = ['huber']
df = pd.DataFrame()
for loss_type in loss_types:
    if loss_type == 'huber':
        delta = 1.345
    elif loss_type == 'logcosh':
        delta = 1.2047
    else:
        delta = 1.4811
    df = pd.concat([df, bootstrap_band(etas, n1=n1, n2=n2, B=B, delta=delta, alpha=0.05, loss_type=loss_type, seed=53)])

In [ ]:
df

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_context('talk')
sns.set_style("whitegrid", {'grid.color': '#F2F2F2'})  # e.g. “darkgrid”, “ticks”, “white”
palette = sns.color_palette("husl", 2)

# Ensure bpb is list-like for explode (np.ndarray → list)
if isinstance(df["bpb"].iloc[0], np.ndarray):
    df["bpb"] = df["bpb"].apply(lambda a: a.tolist())

# Long frame of bootstrap points
bpb_long = (
    df.loc[:, ["eta", "bpb", "loss"]]
      .explode("bpb")
      .rename(columns={"bpb": "boot_bpb"})
)
bpb_long["boot_bpb"] = bpb_long["boot_bpb"].astype(float)

# Sort for a proper ribbon
d = df.sort_values("eta")

# three plots for each loss type
for i, loss_type in enumerate(loss_types):
    fig, ax = plt.subplots(figsize=(8, 5))
    df_loss = df[df["loss"] == loss_type]
    sns.lineplot(data=df_loss, x="eta", y="bp", ax=ax, color=palette[1])
    ax.fill_between(df_loss["eta"], df_loss["bp_lower"], df_loss["bp_upper"], alpha=0.2, color=palette[1])
    sns.scatterplot(data=bpb_long[bpb_long["eta"].isin(etas) & (bpb_long["loss"] == loss_type)], x="eta", y="boot_bpb",
                s=15, alpha=0.2, edgecolor=None, ax=ax, zorder=2, color=palette[0])
    ax.set_xlabel(r"$\eta$")
    ax.set_ylabel(r"$\text{BP}_\eta$")
    plt.savefig(f'bootstrap_ribbon_{loss_type}.pdf', bbox_inches='tight')